# EfficientNet-B0 — 5-Fold Cross-Validation Ensemble

Trains **5 independent B0 models**, one per fold, then averages their logits at test time.
This eliminates the bias introduced by using a fixed fold-3 validation split and maximises
the amount of labelled data seen during training.

Setup: Entropy filtering + Focal-Ordinal loss (same as best single-fold baseline).

In [ ]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import albumentations as Albu
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('../../../')
from utils.dataset import PandasDataset
from utils.models import EfficientNetApi
from utils.metrics import calculate_metrics, format_metrics

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE    = 3
NUM_WORKERS   = 4
OUTPUT_CLASSES = 5          # ordinal thresholds for ISUP 0-5
INIT_LR       = 3e-4
WARMUP_FACTOR = 2
WARMUP_EPOCHS = 1
N_EPOCHS      = 50
DROPOUT_RATE  = 0.6
PATIENCE      = 7
N_FOLDS       = 5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Paths ─────────────────────────────────────────────────────────────
ROOT_DIR   = '../../..'
DATA_DIR   = '../../../..'
IMAGES_DIR = os.path.join(DATA_DIR, 'tiles')

os.makedirs('logs',   exist_ok=True)
os.makedirs('models', exist_ok=True)

In [ ]:
class FocalOrdinalLoss(nn.Module):
    """BCE + focal weighting + ordinal regression term (same as best baseline)."""
    def __init__(self, alpha=0.25, gamma=2.0, ordinal_weight=1.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ordinal_weight = ordinal_weight

    def forward(self, logits, targets):
        targets = targets.to(logits.device).float()
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal = self.alpha * ((1 - pt) ** self.gamma) * bce
        focal_loss = focal.mean()

        expected  = probs.sum(dim=1)
        target_cl = targets.sum(dim=1)
        ordinal_loss = ((expected - target_cl) ** 2).mean() / (logits.shape[1] ** 2)

        return focal_loss + self.ordinal_weight * ordinal_loss

LOSS_FN = FocalOrdinalLoss()
print('Loss function ready')

In [ ]:
# ── Data loading ──────────────────────────────────────────────────────
df_all = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_all.columns = df_all.columns.str.strip()
print(f'Total samples: {len(df_all)}')

# Entropy filtering (remove top-20% hardest samples)
df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
df_entropy_sorted = df_entropy.sort_values('difficulty_score', ascending=False)
n_remove = int(len(df_entropy) * 0.2)
hard_ids = set(df_entropy_sorted.head(n_remove)['image_id'])
df_all = df_all[~df_all['image_id'].isin(hard_ids)].reset_index(drop=True)
print(f'After entropy filtering: {len(df_all)}')

df_test = pd.read_csv(f'{ROOT_DIR}/data/test.csv')

def keep_existing(df, images_dir, ext='png'):
    mask = df['image_id'].apply(lambda x: os.path.isfile(f'{images_dir}/{x}.{ext}'))
    return df[mask].reset_index(drop=True)

df_all  = keep_existing(df_all,  IMAGES_DIR)
df_test = keep_existing(df_test, IMAGES_DIR)
print(f'Trainval available: {len(df_all)}')
print(f'Test available: {len(df_test)}')

In [ ]:
TRAIN_TRANSFORMS = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

def train_one_fold(fold_idx):
    print(f'\n{"="*70}')
    print(f'  FOLD {fold_idx}  (validation = fold {fold_idx})')
    print(f'{"="*70}')

    df_tr = df_all[df_all['fold'] != fold_idx].reset_index(drop=True)
    df_va = df_all[df_all['fold'] == fold_idx].reset_index(drop=True)

    train_ds = PandasDataset(IMAGES_DIR, df_tr, transforms=TRAIN_TRANSFORMS, format='png')
    valid_ds = PandasDataset(IMAGES_DIR, df_va, transforms=None,             format='png')

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          sampler=RandomSampler(train_ds))
    valid_dl = DataLoader(valid_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          sampler=RandomSampler(valid_ds))

    backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model = EfficientNetApi(model=backbone, output_dimensions=OUTPUT_CLASSES,
                            dropout_rate=DROPOUT_RATE).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=INIT_LR / WARMUP_FACTOR)
    sched_cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_EPOCHS - WARMUP_EPOCHS)
    scheduler = GradualWarmupScheduler(optimizer, multiplier=WARMUP_FACTOR,
                                       total_epoch=WARMUP_EPOCHS, after_scheduler=sched_cos)

    model_path = f'models/b0-5fold-fold{fold_idx}.pth'
    log_path   = f'logs/b0-5fold-fold{fold_idx}.txt'

    best_kappa = 0.0
    no_improve = 0

    for epoch in range(1, N_EPOCHS + 1):
        # ── train ──
        model.train()
        train_losses = []
        for batch_x, batch_y, _ in tqdm(train_dl, desc=f'Fold {fold_idx} Ep {epoch} train'):
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            loss = LOSS_FN(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # ── validate ──
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch_x, batch_y, _ in valid_dl:
                logits = model(batch_x.to(DEVICE))
                p = (torch.sigmoid(logits) > 0.5).sum(1).cpu()
                t = batch_y.sum(1).long().cpu()
                preds.append(p); targets.append(t)
        preds   = torch.cat(preds).numpy()
        targets = torch.cat(targets).numpy()
        kappa = cohen_kappa_score(targets, preds, weights='quadratic')

        lr = optimizer.param_groups[0]['lr']
        log = (f'epoch: {epoch} | lr: {lr:.7f} | train_loss: {np.mean(train_losses):.5f} '
               f'| val_kappa: {kappa:.4f}')
        print(log)
        with open(log_path, 'a') as f:
            f.write(log + '\n')

        scheduler.step()

        if kappa >= best_kappa:
            best_kappa = kappa
            no_improve = 0
            torch.save(model.state_dict(), model_path)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'Early stop at epoch {epoch}. Best kappa: {best_kappa:.4f}')
                break

    print(f'Fold {fold_idx} done. Best kappa: {best_kappa:.4f}')
    return model_path, best_kappa

In [ ]:
# ── Train all 5 folds ────────────────────────────────────────────────
fold_results = {}
for fold in range(N_FOLDS):
    path, kappa = train_one_fold(fold)
    fold_results[fold] = {'path': path, 'val_kappa': kappa}

print('\n=== Fold Summary ===')
for f, r in fold_results.items():
    print(f'  Fold {f}: val_kappa = {r["val_kappa"]:.4f}  →  {r["path"]}')
mean_kappa = np.mean([r['val_kappa'] for r in fold_results.values()])
print(f'  CV mean kappa: {mean_kappa:.4f}')

In [ ]:
# ── 5-Fold Ensemble Evaluation on Test Set ───────────────────────────
test_ds = PandasDataset(IMAGES_DIR, df_test, transforms=None, format='png')
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)

def load_fold_model(path):
    backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    m = EfficientNetApi(model=backbone, output_dimensions=OUTPUT_CLASSES,
                        dropout_rate=DROPOUT_RATE).to(DEVICE)
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    return m

fold_models = [load_fold_model(fold_results[f]['path']) for f in range(N_FOLDS)]
print(f'Loaded {len(fold_models)} fold models')

# Average logits across all 5 folds
all_preds, all_targets = [], []
with torch.no_grad():
    for batch_x, batch_y, _ in tqdm(test_dl, desc='Ensemble inference'):
        batch_x = batch_x.to(DEVICE)
        # stack logits from all folds: (N_FOLDS, B, C)
        logits_stack = torch.stack([m(batch_x) for m in fold_models], dim=0)
        mean_logits  = logits_stack.mean(dim=0)              # (B, C)
        preds = (torch.sigmoid(mean_logits) > 0.5).sum(1).cpu()
        targets = batch_y.sum(1).long().cpu()
        all_preds.append(preds)
        all_targets.append(targets)

all_preds   = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()

In [ ]:
# ── Bootstrap metrics ─────────────────────────────────────────────────
metrics = calculate_metrics(all_preds, all_targets)
result  = format_metrics(metrics)
print('\n=== 5-FOLD CV ENSEMBLE — TEST SET RESULTS ===')
print(result)

with open('logs/b0-5fold-cv-test-results.txt', 'w') as f:
    f.write('EfficientNet-B0 — 5-Fold CV Ensemble\n')
    f.write('=' * 70 + '\n\n')
    f.write(f'CV mean val kappa: {mean_kappa:.4f}\n\n')
    f.write(result + '\n\n')
    f.write('Classification Report:\n')
    f.write(classification_report(all_targets, all_preds,
                                  target_names=[f'ISUP {i}' for i in range(6)]))
    cm = confusion_matrix(all_targets, all_preds)
    f.write('\nConfusion Matrix:\n')
    f.write(str(cm))

print('\nResults saved → logs/b0-5fold-cv-test-results.txt')

In [ ]:
# ── Confusion matrix plot ─────────────────────────────────────────────
cm      = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
labels  = [f'ISUP {i}' for i in range(6)]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('B0 5-Fold CV Ensemble — Normalised Confusion Matrix')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('logs/b0-5fold-cv-confusion-matrix-normalized.png', dpi=300)
plt.show()